# LangGraph 01 · 基础图与状态（StateGraph）

这是整个 `01_langgraph/` 的第一课，**同时是全章的开篇**。它回答两个问题：

1. **框架层面**：LangGraph / LangChain / DeepAgents 到底谁是谁？（开篇 · 框架总览）
2. **图层面**：怎么亲手拼出第一张能跑的图，并理解控制流的各种写法？（第 1~4 节）

全章反复出现的四个核心概念，本课都会第一次见面：

| 概念 | 是什么 | 本课的代码形态 |
|---|---|---|
| 状态 State | 在图里流转的共享数据 | `class MyState(TypedDict)` |
| 节点 Node | 一个函数：收 state → 返回增量 | `step_one` / `step_two` |
| 普通边 Edge | 固定从 A 跳到 B | `builder.add_edge("step_one", "step_two")` |
| 条件边 Conditional Edge | 运行时按返回值决定去哪 | `builder.add_conditional_edges(...)` |

> **本 notebook 由 `Agent/01_langgraph/` 下 4 个脚本合并而成**：
> - `00_框架总览_jxsd.py`（全章开篇：三层框架关系，440 行）
> - `01_基础图.py`（课案原版最短实现，80 行）
> - `01_基础图_jxsd.py`（完整版，275 行）
> - `10_控制流与函数式API_官方补充.py`（Send / Command / 函数式 API，347 行）

**官方文档**
- 图 API 参考：<https://docs.langchain.com/oss/python/langgraph/graph-api>
- 快速上手：<https://docs.langchain.com/oss/python/langgraph/quickstart>
- 用图 API 构建：<https://docs.langchain.com/oss/python/langgraph/use-graph-api>
- 函数式 API：<https://docs.langchain.com/oss/python/langgraph/functional-api>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟢 运行档位 | **离线可跑** —— 全程不发起任何大模型调用、不起服务 |
| 依赖 | `langgraph` / `langchain` / `deepagents`（venv 已装） |
| 密钥 | 无（开篇会读取一次 `config.settings` 构造模型客户端对象，但**不 invoke**） |
| 前置服务 | 无 |
| 预计耗时 | 约 5 秒 |

> ⚠️ 关于档位的说明：本课合并的 `00_框架总览_jxsd.py` 原文里有两次**真实模型提问**。
> 为了让整课保持 🟢 离线可跑，那两段对话被封装成了**不被调用的函数**（开篇 · 6），
> 只保留「建图、看结构」的部分 —— 想看真实对话请跑归档脚本
> `Agent/_py_source/01_langgraph/00_框架总览_jxsd.py`。

## 本节地图

本课从「框架」到「图」到「控制流」一共五块：

| 块 | 内容 | 来源 |
|---|---|---|
| 开篇 | 框架总览：LangGraph / LangChain / DeepAgents 三层关系 | `00_框架总览_jxsd.py` |
| 1 | 课案原版：最小的一张图 | `01_基础图.py` |
| 2 | 完整版：每个元素正式定义一遍 | `01_基础图_jxsd.py` |
| 3 | 追加式字段：`Annotated` + `operator.add` | `01_基础图_jxsd.py` |
| 4 | 控制流三件套与函数式 API（Send / Command / @task） | `10_控制流与函数式API_官方补充.py` |

先看第 1 节里那张图的数据流（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

```mermaid
graph LR
    A["START<br/>图入口"] --> B["step_one<br/>count + 1"]
    B --> C["step_two<br/>count × 2"]
    C --> D{"choose_path<br/>count > 10 ?"}
    D -->|"是"| E["big<br/>太大了"]
    D -->|"否"| F["small<br/>很小"]
    E --> G["END<br/>图出口"]
    F --> G
```

上面这张图等价于下面这张表：

| 从 | 到 | 边的类型 | 什么时候走 |
|---|---|---|---|
| `START` | `step_one` | 普通边 | 总是 |
| `step_one` | `step_two` | 普通边 | 总是 |
| `step_two` | `big` 或 `small` | **条件边** | 由 `choose_path` 的返回值决定 |
| `big` / `small` | `END` | 普通边 | 总是 |

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课其实用不到 `config`，但这一格仍然保留 —— 一是保持全仓统一，
> 二是它顺便给出了 `NB_DIR` / `WORKDIR` 两个变量，后续课时要用它来定位临时文件。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 开篇 · 框架总览：LangGraph / LangChain / DeepAgents 三层关系

动手写图之前，先回答一个最容易绕晕的问题：「LangGraph、LangChain、DeepAgents 到底谁是谁？」

- **LangGraph** —— 最底层的「图引擎」，只管节点和边的调度。它不认识大模型、不认识工具，只认识「图」。
- **LangChain** —— 在 LangGraph 上封装了 `create_agent()`，提供模型 / 工具 / 中间件（Middleware）等现成组件。
  `create_agent()` 的本质：动态拼一张 `StateGraph` 出来。
- **DeepAgents** —— 在 LangChain 的 `create_agent()` 之上再包一层，把「文件操作 / 子 Agent / 任务规划 / 上下文摘要」
  这些中间件预配好，开箱即用。

一句话记忆：`DeepAgents ⊃ create_agent ⊃ 内部拼出的那张 StateGraph`（越往上越省事，越往下越自由）。

这一节做三件事（都是可运行的真代码，不是伪代码）：

1. 打印三层框架的对比表；
2. 把课案里 factory.py 的「伪代码」逐行变成一个**能跑的迷你 ReAct 图**；
3. 用真正的 `create_agent()` / `create_deep_agent()` 编译出图，把节点清单和迷你图逐行对照。

### 三层框架的层次关系：打印对比表

下面先把课案的层次图与 LangChain vs DeepAgents 对比表原样打印出来。
注意 `_dwidth` / `_pad`：中文是全角字符，直接 `f"{s:<12}"` 会顶歪整张表，
所以先按「终端显示宽度」算宽度再补空格。

In [ ]:
import unicodedata

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

from config import settings

# 大模型统一走 config.settings（密钥不落代码，只从 .env 读）。
# 注意：这里只是「构造客户端对象」，不会发起任何网络请求；
# 真正花钱的是 model_callable 里的 invoke()，本课离线档位不会走到那一步。
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def _dwidth(text: str) -> int:
    """按「终端显示宽度」计算字符串宽度：中日韩全角字符占 2 列。"""
    return sum(2 if unicodedata.east_asian_width(ch) in ("W", "F") else 1 for ch in text)


def _pad(text: str, width: int) -> str:
    """把 text 补到指定的显示宽度（左对齐）。"""
    return text + " " * max(0, width - _dwidth(text))


def print_framework_table() -> None:
    print("=" * 78)
    print("① 三者的层次关系：谁包着谁")
    print("=" * 78)
    print(
        """
  ┌──────────────────────── DeepAgents（最上层，开箱即用）────────────────────────┐
  │  create_deep_agent(model)                                                     │
  │    └─ 内部调用 LangChain 的 create_agent()，并预配好一整套中间件与工具：        │
  │       文件操作(ls/read_file/write_file/edit_file) / 子 Agent(task 工具) /      │
  │       任务规划(TodoList) / 上下文自动摘要(Summarization)                       │
  └───────────────────────────────────┬───────────────────────────────────────────┘
                                      │ 封装
  ┌───────────────────────────────────▼───────────────────────────────────────────┐
  │  create_agent(model, tools, middleware)   ← LangChain 提供的「组装工厂」       │
  │    └─ 内部动态构建一张 StateGraph（就是下面那张迷你图）                        │
  └───────────────────────────────────┬───────────────────────────────────────────┘
                                      │ 基于
  ┌───────────────────────────────────▼───────────────────────────────────────────┐
  │  StateGraph / 节点 / 边 / 条件边   ← LangGraph：只管调度，不认识大模型         │
  └───────────────────────────────────────────────────────────────────────────────┘
"""
    )
    print("② 课案原文的 LangChain vs DeepAgents 对比表：")
    rows = [
        ("定位", "组件库，自己组装", "LangChain 的封装，开箱即用"),
        ("创建 Agent", "create_agent(model, tools, middleware)", "create_deep_agent(model) 内部调用 create_agent"),
        ("文件操作", "需要手动加 FilesystemMiddleware", "内置 ls / read_file / write_file / edit_file"),
        ("子 Agent", "需要手动加 SubAgentMiddleware", "内置 task 工具委派子 Agent"),
        ("任务规划", "需要手动加 TodoListMiddleware", "内置自动拆解任务"),
        ("上下文管理", "需要手动加 SummarizationMiddleware", "内置自动摘要压缩"),
        ("适用场景", "需要精细控制每个组件", "快速搭建，想立刻干活"),
    ]
    print(f"  {_pad('', 12)}{_pad('LangChain', 40)}{_pad('DeepAgents', 42)}")
    print("  " + "-" * 94)
    for name, lc, da in rows:
        print(f"  {_pad(name, 12)}{_pad(lc, 40)}{_pad(da, 42)}")
    print()

In [ ]:
print_framework_table()

### 预期输出

```text
==============================================================================
① 三者的层次关系：谁包着谁
==============================================================================

  ┌──────────────────────── DeepAgents（最上层，开箱即用）────────────────────────┐
  │  create_deep_agent(model)                                                     │
  │    └─ 内部调用 LangChain 的 create_agent()，并预配好一整套中间件与工具：        │
  │       文件操作(ls/read_file/write_file/edit_file) / 子 Agent(task 工具) /      │
  │       任务规划(TodoList) / 上下文自动摘要(Summarization)                       │
  └───────────────────────────────────┬───────────────────────────────────────────┘
                                      │ 封装
  ┌───────────────────────────────────▼───────────────────────────────────────────┐
  │  create_agent(model, tools, middleware)   ← LangChain 提供的「组装工厂」       │
  │    └─ 内部动态构建一张 StateGraph（就是下面那张迷你图）                        │
  └───────────────────────────────────┬───────────────────────────────────────────┘
                                      │ 基于
  ┌───────────────────────────────────▼───────────────────────────────────────────┐
  │  StateGraph / 节点 / 边 / 条件边   ← LangGraph：只管调度，不认识大模型         │
  └───────────────────────────────────────────────────────────────────────────────┘

② 课案原文的 LangChain vs DeepAgents 对比表：
              LangChain                               DeepAgents                                
  ----------------------------------------------------------------------------------------------
  定位        组件库，自己组装                        LangChain 的封装，开箱即用                
  创建 Agent  create_agent(model, tools, middleware)  create_deep_agent(model) 内部调用 create_agent
  文件操作    需要手动加 FilesystemMiddleware         内置 ls / read_file / write_file / edit_file
  子 Agent    需要手动加 SubAgentMiddleware           内置 task 工具委派子 Agent                
  任务规划    需要手动加 TodoListMiddleware           内置自动拆解任务                          
  上下文管理  需要手动加 SummarizationMiddleware      内置自动摘要压缩                          
  适用场景    需要精细控制每个组件                    快速搭建，想立刻干活                      
```

### 工具与大模型：迷你 ReAct 图的零件

迷你 ReAct 图需要两样零件：一个「模型能看懂的工具」和一个「绑了工具的模型」。

- `@tool` 装饰器把普通函数变成工具：函数名 → 工具名，docstring → 工具描述，
  类型标注 → 参数 schema（Pydantic 自动生成）；
- `bind_tools` 把工具清单挂到模型上，之后模型返回的 `AIMessage` 里就可能带
  `tool_calls`（要调哪个工具、参数是什么）。

In [ ]:
@tool
def get_weather(city: str) -> str:
    """查询指定城市的当前天气。参数 city 是城市名，例如「北京」。"""
    return f"{city}：晴，25℃，微风"


# bind_tools：把工具清单「挂」到模型上
llm_with_tools = llm.bind_tools([get_weather])

### 把课案伪代码变成能跑的迷你 ReAct 图

课案这一节只给了 factory.py 的**伪代码**（没有可运行文件）。下面把它的 6 步
落成真实可跑的图。其中「中间件钩子」分两种，务必分清：

| 钩子类型 | 落地方式 | 是否占图节点 |
|---|---|---|
| 节点式钩子（before_agent / before_model / after_model / after_agent） | 注册成独立图节点 | 是 |
| 包裹式钩子（wrap_model_call / wrap_tool_call） | 函数嵌套（装饰器） | 否 |

In [ ]:
# ---------- 节点函数：model（调用 LLM）----------
def model_callable(state: MessagesState) -> dict:
    """model 节点：把当前消息列表交给模型，返回模型的新消息。"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# ---------- 节点式钩子：注册成独立图节点 ----------
def demo_before_agent(state: MessagesState) -> dict:
    print("      [节点式钩子] demo.before_agent  —— Agent 启动前，只跑一次")
    return {}


def demo_before_model(state: MessagesState) -> dict:
    print(f"      [节点式钩子] demo.before_model   —— 调模型前，当前 {len(state['messages'])} 条消息")
    return {}


def demo_after_model(state: MessagesState) -> dict:
    last = state["messages"][-1]
    calls = getattr(last, "tool_calls", None) or []
    print(f"      [节点式钩子] demo.after_model    —— 模型返回，tool_calls={[c['name'] for c in calls]}")
    return {}


def demo_after_agent(state: MessagesState) -> dict:
    print("      [节点式钩子] demo.after_agent    —— Agent 结束前，只跑一次")
    return {}


# ---------- 包裹式钩子：函数嵌套，不占图节点 ----------
def wrap_model_call(fn):
    """包裹式钩子：接收一个节点函数，返回「加了前后逻辑」的新函数。"""
    def wrapped(state: MessagesState) -> dict:
        print("      [包裹式钩子] wrap_model_call  —— 进入 model 节点之前")
        result = fn(state)
        print("      [包裹式钩子] wrap_model_call  —— model 节点返回之后")
        return result
    return wrapped


# ---------- 路由函数：模型返回 tool_calls → tools，否则 → 结束 ----------
def route_after_model(state: MessagesState) -> str:
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return "exit"


# ---------- 组装图（严格照伪代码的 4 步）----------
def build_mini_react_graph():
    builder = StateGraph(MessagesState)
    builder.add_node("model", wrap_model_call(model_callable))
    builder.add_node("tools", ToolNode([get_weather]))
    builder.add_node("demo.before_agent", demo_before_agent)
    builder.add_node("demo.before_model", demo_before_model)
    builder.add_node("demo.after_model", demo_after_model)
    builder.add_node("demo.after_agent", demo_after_agent)
    builder.add_edge(START, "demo.before_agent")
    builder.add_edge("demo.before_agent", "demo.before_model")
    builder.add_edge("demo.before_model", "model")
    builder.add_edge("model", "demo.after_model")
    builder.add_conditional_edges(
        "demo.after_model",
        route_after_model,
        {"tools": "tools", "exit": "demo.after_agent"},
    )
    builder.add_edge("tools", "demo.before_model")
    builder.add_edge("demo.after_agent", END)
    return builder.compile()


def dump_graph(graph, title: str) -> None:
    """打印一张编译好的图的节点与边，用来做「真身 vs 迷你图」的对照。"""
    net = graph.get_graph()
    print(f"  {title}")
    print(f"    节点：{sorted(net.nodes.keys())}")
    for e in net.edges:
        cond = "  （条件边）" if e.conditional else ""
        print(f"    边  ：{e.source:<34} → {e.target}{cond}")

### 组装并打印迷你图结构（离线可看）

只「建图 + 看结构」，**不调用模型**，所以这一步离线秒出。
注意节点清单里出现了 `demo.before_agent` 这类钩子 —— 它们是**真实的图节点**。

In [ ]:
print("=" * 78)
print("③ 把课案伪代码变成能跑的迷你 ReAct 图")
print("=" * 78)
mini_graph = build_mini_react_graph()
dump_graph(mini_graph, "迷你图的真实结构（注意：钩子是独立节点）")
print()

### 预期输出

```text
==============================================================================
③ 把课案伪代码变成能跑的迷你 ReAct 图
==============================================================================
  迷你图的真实结构（注意：钩子是独立节点）
    节点：['__end__', '__start__', 'demo.after_agent', 'demo.after_model', 'demo.before_agent', 'demo.before_model', 'model', 'tools']
    边  ：__start__                          → demo.before_agent
    边  ：demo.after_model                   → demo.after_agent  （条件边）
    边  ：demo.after_model                   → tools  （条件边）
    边  ：demo.before_agent                  → demo.before_model
    边  ：demo.before_model                  → model
    边  ：model                              → demo.after_model
    边  ：tools                              → demo.before_model
    边  ：demo.after_agent                   → __end__
```

### 与真身对照：create_agent / create_deep_agent 编译出的图

这一步同样只「建图 + 看结构」，不发起对话，所以照样离线。对照结论：
真身把中间件钩子注册成了 `DemoMiddleware.before_agent` 这类**独立节点**，
和迷你图里的 `demo.before_agent` 是同一个套路 —— 差别只是名字前缀 `{中间件类名}.{钩子名}`。

> （不同 langchain / deepagents 版本的节点命名可能略有差异，以当前环境打印为准。）

In [ ]:
print("=" * 78)
print("④ 对照实验：真正的 create_agent() 编译出来的图长什么样")
print("=" * 78)
try:
    from langchain.agents import create_agent
    from langchain.agents.middleware import AgentMiddleware

    class DemoMiddleware(AgentMiddleware):
        """中间件类：四个钩子方法的签名和课案伪代码一致。"""
        def before_agent(self, state, runtime):
            return None
        def before_model(self, state, runtime):
            return None
        def after_model(self, state, runtime):
            return None
        def after_agent(self, state, runtime):
            return None

    real_agent = create_agent(
        model=llm,
        tools=[get_weather],
        middleware=[DemoMiddleware()],
    )
    dump_graph(real_agent, "create_agent(model, tools, middleware) 编译出的图")
    print(
        "\n  对照结论：真身把中间件钩子注册成了 "
        "`DemoMiddleware.before_agent` 这类**独立节点**，\n"
        "  和我们迷你图里的 `demo.before_agent` 是同一个套路——\n"
        "  唯一的差别只是名字前缀：`{中间件类名}.{钩子名}`。\n"
        "  而包裹式钩子（wrap_model_call / wrap_tool_call）在图里**看不到**，\n"
        "  因为它们走的是函数嵌套，不占节点。"
    )
except Exception as exc:
    print(f"  跳过 create_agent 对照（当前环境不可用）：{type(exc).__name__}: {exc}")
print()

### 预期输出

```text
==============================================================================
④ 对照实验：真正的 create_agent() 编译出来的图长什么样
==============================================================================
  create_agent(model, tools, middleware) 编译出的图
    节点：['DemoMiddleware.after_agent', 'DemoMiddleware.after_model', 'DemoMiddleware.before_agent', 'DemoMiddleware.before_model', '__end__', '__start__', 'model', 'tools']
    边  ：DemoMiddleware.after_model         → DemoMiddleware.after_agent  （条件边）
    边  ：DemoMiddleware.after_model         → DemoMiddleware.before_model  （条件边）
    边  ：DemoMiddleware.after_model         → tools  （条件边）
    边  ：DemoMiddleware.before_agent        → DemoMiddleware.before_model
    边  ：DemoMiddleware.before_model        → model
    边  ：__start__                          → DemoMiddleware.before_agent
    边  ：model                              → DemoMiddleware.after_model
    边  ：tools                              → DemoMiddleware.before_model  （条件边）
    边  ：DemoMiddleware.after_agent         → __end__

  对照结论：真身把中间件钩子注册成了 `DemoMiddleware.before_agent` 这类**独立节点**，
  和我们迷你图里的 `demo.before_agent` 是同一个套路——
  唯一的差别只是名字前缀：`{中间件类名}.{钩子名}`。
  而包裹式钩子（wrap_model_call / wrap_tool_call）在图里**看不到**，
  因为它们走的是函数嵌套，不占节点。
```

In [ ]:
print("=" * 78)
print("⑤ DeepAgents：create_deep_agent() 也是拼一张 StateGraph")
print("=" * 78)
try:
    from deepagents import create_deep_agent

    deep_agent = create_deep_agent(model=llm)
    dump_graph(deep_agent, "create_deep_agent(model) 编译出的图")
    print(
        "\n  注意 `PatchToolCallsMiddleware.before_agent` 这个节点：\n"
        "  它就是 DeepAgents 帮你预配好的中间件自动注册出来的图节点——\n"
        "  这正是课案那句话的实证：create_deep_agent() 内部调用了\n"
        "  LangChain 的 create_agent()，所以最终产物同样是 LangGraph 的图。"
    )
except ImportError:
    print("  未安装 deepagents，跳过本段演示。安装命令：uv add deepagents")
print()
print("=" * 78)
print("全章结论：不管用哪一层，跑起来的东西最终都是 LangGraph 的一张图。")
print("=" * 78)

### 预期输出

```text
==============================================================================
⑤ DeepAgents：create_deep_agent() 也是拼一张 StateGraph
==============================================================================
  create_deep_agent(model) 编译出的图
    节点：['PatchToolCallsMiddleware.before_agent', '__end__', '__start__', 'model', 'tools']
    边  ：PatchToolCallsMiddleware.before_agent → model
    边  ：__start__                          → PatchToolCallsMiddleware.before_agent
    边  ：model                              → __end__  （条件边）
    边  ：model                              → model  （条件边）
    边  ：model                              → tools  （条件边）
    边  ：tools                              → model  （条件边）

  注意 `PatchToolCallsMiddleware.before_agent` 这个节点：
  它就是 DeepAgents 帮你预配好的中间件自动注册出来的图节点——
  这正是课案那句话的实证：create_deep_agent() 内部调用了
  LangChain 的 create_agent()，所以最终产物同样是 LangGraph 的图。

==============================================================================
全章结论：不管用哪一层，跑起来的东西最终都是 LangGraph 的一张图。
==============================================================================
```

### 需要模型的两段对话（本课离线档位下不调用）

源文件 `00_框架总览_jxsd.py` 里还有两次**真实模型提问**：一次「北京天气」
（模型会调 `get_weather`，走完整 ReAct 循环），一次「1+1」证明不走工具也能直答。
为了让本课保持 🟢 离线可跑，把这两段封装成下面这个**不被调用**的函数：
想亲眼看真实对话，去掉最后那行注释（改成 `run_mini_graph_demo(mini_graph)`），
或直接跑归档脚本 `Agent/_py_source/01_langgraph/00_框架总览_jxsd.py`。

In [ ]:
def run_mini_graph_demo(mini_graph) -> None:
    """需要大模型的两段对话演示（本 notebook 离线档位下不会调用）。"""
    print("── 提问：北京今天天气怎么样？（模型需要调工具，会走 ReAct 循环）")
    result = mini_graph.invoke(
        {"messages": [{"role": "user", "content": "北京今天天气怎么样？"}]}
    )
    print(f"  最终回复：{result['messages'][-1].content}")
    print(f"  消息总数：{len(result['messages'])} 条"
          f"（user → AI(带 tool_calls) → ToolMessage → AI(最终答复)）")
    print()

    print("── 提问：1+1 等于几？（不需要工具，模型直接回答，不走 tools）")
    result = mini_graph.invoke({"messages": [{"role": "user", "content": "1+1 等于几？只回数字"}]})
    print(f"  最终回复：{result['messages'][-1].content}")
    print()


# 本 notebook 为 🟢 离线档位，不调用上面的函数（去掉下面这行注释即可真正跑模型）：
# run_mini_graph_demo(mini_graph)

## 1. 课案原版：最小的一张图

课案原版只有 80 行，正好够把四个概念各出现一次。
**先看最短的实现，再看完整版**，两者的差距就是本课要讲的全部内容。

In [ ]:
# ---------- 1.1 状态：一个普通的 TypedDict ----------
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    """
    状态就是一个 TypedDict，所有节点共享。

    - messages：普通覆盖式字段（后写的值直接覆盖先写的）
    - history ：使用 Annotated + operator.add 实现「追加式」字段
                每个节点返回的列表会被拼接（reduce）到原有列表后面
    """
    messages: str
    history: Annotated[list[str], operator.add]

In [ ]:
# ---------- 1.2 节点：就是普通函数 ----------
def node_a(state: State) -> dict:
    """节点 A：接收当前状态，返回对状态的「增量更新」（而不是全量状态）"""
    print(f"[节点A] 收到消息：{state['messages']}")
    # 只需要返回要更新的字段
    return {"messages": "A 已处理", "history": ["经过节点A"]}


def node_b(state: State) -> dict:
    """节点 B：可以读到前面节点写入的数据"""
    print(f"[节点B] 收到消息：{state['messages']}")
    return {"messages": "B 已处理", "history": ["经过节点B"]}

In [ ]:
# ---------- 1.3 条件边：根据状态动态决定下一个节点 ----------
def route(state: State) -> str:
    """返回值是「下一个节点」的名字，也可以返回 END 表示结束"""
    if "B" in state["messages"]:
        return END
    return "b"

In [ ]:
# ---------- 1.4 组装图 ----------
builder = StateGraph(State)

# 注册节点
builder.add_node("a", node_a)
builder.add_node("b", node_b)

# 固定边：START -> a
builder.add_edge(START, "a")
# 条件边：a 之后走 route 函数决定去向
builder.add_conditional_edges("a", route, ["b", END])
# 固定边：b -> END
builder.add_edge("b", END)

# 编译：把 builder 变成可执行的图
graph = builder.compile()

In [ ]:
# ---------- 1.5 执行 ----------
# invoke：一次性执行完整张图，返回最终状态
result = graph.invoke({"messages": "你好"})
print("最终状态：", result)

### 预期输出

```text
[节点A] 收到消息：你好
[节点B] 收到消息：A 已处理
最终状态： {'messages': 'B 已处理', 'history': ['经过节点A', '经过节点B']}
```

**三个值得停一下看的细节：**

1. 节点 B 收到的是 `"A 已处理"` 而不是 `"你好"` —— 说明**节点 A 的返回值真的并进了状态**；
2. `history` 里两条都在 —— 因为字段声明用了 `Annotated[..., operator.add]`（追加式），
   而 `messages` 只剩最后一条（覆盖式）；
3. 条件边走的是 `"b"` 而不是 `END` —— 因为此刻 `state["messages"]` 是 `"A 已处理"`，
   里面不含字母 `"B"`。

## 2. 完整版：每个元素都正式定义一遍

课案原版把状态、节点、边混在一个文件里；完整版把它们**按职责切开**，
这样后面加节点、加分支时不会互相干扰。

### 2.1 状态：图的「共享内存」

`TypedDict` 只描述「有哪些字段、什么类型」，**不产生运行时校验**。
它的真正作用有两个：

- 给编辑器做补全与类型检查；
- 告诉 LangGraph **每个字段该怎么合并**（这才是关键，见 2.5 与第 3 节）。

In [ ]:
# ============================================================
# 1. 定义状态（图中流转的数据）
# ============================================================
class MyState(TypedDict):
    """课案原文的状态定义。

    TypedDict 只描述「有哪些字段、什么类型」，不产生运行时校验，
    它的作用是给编辑器 / LangGraph 提供字段清单和合并规则。
    """

    count: int  # 计数器：普通字段，节点返回什么就是什么（覆盖式）
    log: str  # 日志：普通字段，每个节点覆盖写入

### 2.2 节点：返回「增量」，不是「全量」

这是新手最容易搞错的一点：节点函数**只需要返回自己改动的字段**。
返回完整 state 不会报错，但会让 reducer 的语义变得难以预料。

In [ ]:
# ============================================================
# 2. 定义节点函数（接收 state，返回更新的 state）
# ============================================================
def step_one(state: MyState) -> dict:
    """步骤一：count + 1，并把日志重置为「执行了步骤一」。"""
    # 注意只返回要改的字段。log 这里是覆盖，不是拼接——
    # 想拼接（保留历史）就得用 Annotated + operator.add，见第 5 节。
    return {"count": state["count"] + 1, "log": "执行了步骤一"}


def step_two(state: MyState) -> dict:
    """步骤二：count × 2，并把新日志拼在旧日志后面（手工拼字符串）。"""
    return {"count": state["count"] * 2, "log": state["log"] + " → 步骤二"}

### 2.3 路由函数：只返回「下一个节点的名字」

⚠️ **路由函数不执行跳转，也不改状态**。真正查表跳转的是 LangGraph 自己。
这条边界想清楚了，条件边就不会写错。

In [ ]:
def choose_path(state: MyState) -> str:
    """条件边的路由函数。

    ⚠️ 路由函数只做一件事：**返回「下一个节点的名字」**（字符串）。
    它不执行跳转，也不碰状态——真正的跳转由 LangGraph 按
    add_conditional_edges 里给的映射表完成。
    """
    # 条件边：根据 count 值决定走哪条路
    return "big" if state["count"] > 10 else "small"


def big_handler(state: MyState) -> dict:
    """大数分支：count > 10 时走这里。"""
    return {"log": f"count={state['count']}, 太大了"}


def small_handler(state: MyState) -> dict:
    """小数分支：count <= 10 时走这里。"""
    return {"log": f"count={state['count']}, 很小"}

### 2.4 组装图

`add_edge(START, "step_one")` 里的 `START` / `END` 是 LangGraph 的**虚拟节点**：
它们不对应任何函数，只标记「从哪进、到哪出」。

In [ ]:
# ============================================================
# 3. 构建图（课案原文的组装流程）
# ============================================================
builder = StateGraph(MyState)

builder.add_node("step_one", step_one)  # 添加节点：节点名 "step_one" → 函数 step_one
builder.add_node("step_two", step_two)
builder.add_node("big", big_handler)
builder.add_node("small", small_handler)

builder.add_edge(START, "step_one")  # 普通边：START → step_one（START 是图的虚拟入口）
builder.add_edge("step_one", "step_two")  # 普通边：step_one → step_two
builder.add_edge("big", END)  # big → 结束（END 是图的虚拟出口）
builder.add_edge("small", END)  # small → 结束

### 2.5 条件边：本节的绝对重点

`add_conditional_edges` 的第三个参数有**三种写法**，行为并不一样：

| 写法 | 例子 | 语义 |
|---|---|---|
| ① 列表 | `["big", "small"]` | 路由函数返回 `"big"`，就必须存在**同名节点** `big` |
| ② 映射字典（推荐） | `{"big": "big", "small": "small"}` | 左边是返回值，右边是真正跳的节点名；**两者可以不同名** |
| ③ 不传第三个参数 | — | LangGraph 自行推断目标，少写代码但图结构不直观 |

为什么推荐映射字典？因为它把「路由函数的返回值」和「节点名」解耦了，
例如 `{"big": "handle_large"}` 完全合法 —— 返回值只是**内部的暗号**。

> 第三个参数还有一个隐藏作用：**声明这笔分叉可能去哪些地方**。
> 于是 `graph.get_graph()` 才能把条件边画出来 —— 不传，图就是残缺的。
> 这一点在下面的「打印图结构」里会亲眼看到。

In [ ]:
# ---------- 3.1 条件边：本节的绝对重点 ----------
# add_conditional_edges 的三个参数，课案里给了「列表」写法，
# 这里特意用「映射字典」写法，把两者的区别讲透：
#
#   builder.add_conditional_edges("step_two", choose_path, {"big": "big", "small": "small"})
#                                 ↑ 源节点    ↑ 路由函数   ↑ 路由函数返回值 → 真实节点名
builder.add_conditional_edges(
    "step_two",
    choose_path,  # 路由函数，返回目标节点名
    {"big": "big", "small": "small"},  # 返回值 → 节点映射
)

graph = builder.compile()  # 编译：把 builder 变成可执行的图（编译后不能再改结构）
print("图已编译：", graph)

### 2.6 跑起来：两条分支都要走一遍

只跑一条分支是看不出条件边生效的 —— 换个初值，让 `count` 越过 10，走另一条路。

In [ ]:
# ============================================================
# 4. 运行（课案原文的 invoke）
# ============================================================
def run_course_example() -> None:
    """课案原文示例：count=1 → +1=2 → ×2=4 → small → "count=4, 很小"。"""
    print("=" * 72)
    print("① 课案原文示例：invoke({'count': 1, 'log': ''})")
    print("=" * 72)
    print("  数据流：count=1 —step_one→ 2 —step_two→ 4 —choose_path→ small")
    result = graph.invoke({"count": 1, "log": ""})
    print("  返回：", result)
    # 预期输出：{'count': 4, 'log': 'count=4, 很小'}
    print()


run_course_example()

### 预期输出

```text
========================================================================
① 课案原文示例：invoke({'count': 1, 'log': ''})
========================================================================
  数据流：count=1 —step_one→ 2 —step_two→ 4 —choose_path→ small
  返回： {'count': 4, 'log': 'count=4, 很小'}
```

课案注释里写的那条数据流（`count=1 → +1=2 → ×2=4 → small`）**本机实测完全一致**。

In [ ]:
def run_big_branch() -> None:
    """换个初值走 big 分支：count=6 → +1=7 → ×2=14 > 10 → big。"""
    print("=" * 72)
    print("② 换一个初值，让条件边走另一条分叉：invoke({'count': 6, 'log': ''})")
    print("=" * 72)
    print("  数据流：count=6 —step_one→ 7 —step_two→ 14 > 10 —choose_path→ big")
    result = graph.invoke({"count": 6, "log": ""})
    print("  返回：", result)
    # 预期输出：{'count': 14, 'log': 'count=14, 太大了'}
    print()


run_big_branch()

### 预期输出

```text
========================================================================
② 换一个初值，让条件边走另一条分叉：invoke({'count': 6, 'log': ''})
========================================================================
  数据流：count=6 —step_one→ 7 —step_two→ 14 > 10 —choose_path→ big
  返回： {'count': 14, 'log': 'count=14, 太大了'}
```

**同一个图、同一段代码，只换了初值，走的就是另一条边** —— 这就是条件边的意义。

In [ ]:
def show_graph_structure() -> None:
    """把图的结构打印出来——让「节点/普通边/条件边」看得见摸得着。"""
    print("=" * 72)
    print("③ 图的结构（get_graph()）：普通边与条件边的区别一目了然")
    print("=" * 72)
    net = graph.get_graph()
    for edge in sorted(net.edges, key=lambda e: (e.source, e.target)):
        kind = "条件边" if edge.conditional else "普通边"
        print(f"  [{kind}] {edge.source:<10} → {edge.target}")
    print()


show_graph_structure()

### 预期输出

```text
========================================================================
③ 图的结构（get_graph()）：普通边与条件边的区别一目了然
========================================================================
  [普通边] __start__  → step_one
  [普通边] big        → __end__
  [普通边] small      → __end__
  [普通边] step_one   → step_two
  [条件边] step_two   → big
  [条件边] step_two   → small
```

这里有两个「原来如此」：

- `START` / `END` 在打印里叫 `__start__` / `__end__` —— 它们是 LangGraph
  内部插入的虚拟节点，所以**永远出现在边表里**，但你没有写进 `add_node`；
- `step_two` 出现了**两条**出边，且都标着「条件边」—— 正是第三个参数
  （那个映射字典）声明出来的。**把第三个参数去掉，这两行就会消失**，
  图变成残缺的。

## 3. 追加式字段：`Annotated` + `operator.add`

上面 `MyState.log` 是**覆盖式**的。之所以还能看到历史，是因为我们在
`step_two` 里**手工**把字符串拼了起来（`state["log"] + " → 步骤二"`）。

节点一多，每个都要手工拼接，又啰嗦又容易漏。LangGraph 的解法是
「声明式 reducer」：给字段挂一个**合并函数**，让框架自动合并。

In [ ]:
# ============================================================
# 5. 补充：Annotated + operator.add —— 「追加式」字段
# ============================================================
class ChatState(TypedDict):
    """演示追加式字段的状态。

    | 写法                                  | 合并行为                | 典型用途          |
    |---------------------------------------|-------------------------|-------------------|
    | title: str                            | 覆盖：新值盖旧值        | 当前状态类字段    |
    | history: Annotated[list, operator.add]| 追加：新旧列表拼接      | 对话历史、执行日志|
    | messages: Annotated[list, add_messages]| 追加 + 按 id 去重/更新 | LangGraph 内置    |

    注意 `operator.add` 作用在 list 上就是 `[] + []`（列表拼接），
    作用在 int/str 上则是数值相加 / 字符串相连——所以 reducer 选错类型会出怪事。
    """

    title: str  # 普通字段：覆盖
    history: Annotated[list[str], operator.add]  # 追加式字段：自动 concat

注意看下面两个节点：**写法与第 2 节的 `node_a` / `node_b` 完全一样**，
都只返回「自己那一条」。区别**只来自状态字段的声明方式**。

In [ ]:
def node_a(state: ChatState) -> dict:
    """节点 A：只返回「我这一条」日志，不操心拼接。"""
    # 对比上面 step_two 的手工拼接——这里只写自己新增的部分
    return {"title": "A 处理后的标题", "history": ["经过节点A"]}


def node_b(state: ChatState) -> dict:
    """节点 B：同样只返回自己那一条。"""
    return {"title": "B 处理后的标题", "history": ["经过节点B"]}


annotated_builder = StateGraph(ChatState)
annotated_builder.add_node("a", node_a)
annotated_builder.add_node("b", node_b)
annotated_builder.add_edge(START, "a")
annotated_builder.add_edge("a", "b")
annotated_builder.add_edge("b", END)
annotated_graph = annotated_builder.compile()

In [ ]:
def run_annotated_demo() -> None:
    print("=" * 72)
    print("④ 补充：Annotated + operator.add 的「追加式」字段")
    print("=" * 72)
    result = annotated_graph.invoke({"title": "初始标题", "history": ["图开始执行"]})
    print("  返回：", result)
    # 预期输出：
    #   title   → "B 处理后的标题"（覆盖式：A 写的被 B 盖掉了，只剩最后一句）
    #   history → ['图开始执行', '经过节点A', '经过节点B']（追加式：一条都没丢）
    print()
    print("  对照结论：同一个状态里，title 走覆盖、history 走追加；")
    print("  差别**只来自字段的声明方式**，节点函数本身的写法完全一样。")
    print()


run_annotated_demo()

### 预期输出

```text
========================================================================
④ 补充：Annotated + operator.add 的「追加式」字段
========================================================================
  返回： {'title': 'B 处理后的标题', 'history': ['图开始执行', '经过节点A', '经过节点B']}

  对照结论：同一个状态里，title 走覆盖、history 走追加；
  差别**只来自字段的声明方式**，节点函数本身的写法完全一样。
```

对照着看这一张表，就明白 reducer 到底做了什么：

| 字段 | 初值 | 节点 A 返回 | 节点 B 返回 | 最终值 | 为什么 |
|---|---|---|---|---|---|
| `title` | `"初始标题"` | `"A 处理后的标题"` | `"B 处理后的标题"` | `"B 处理后的标题"` | 普通字段 → **覆盖** |
| `history` | `['图开始执行']` | `['经过节点A']` | `['经过节点B']` | 三条全在 | `operator.add` → **追加** |

## 4. 控制流三件套与函数式 API（官方补充）

前三节把「图 API」的主干讲完了：StateGraph、节点、边、条件边、reducer。
官方文档还有三个高频机制课案没覆盖，这里补上，全部离线可跑（0 次模型调用）：

| 机制 | 解决什么问题 | 对应官方出处 |
|---|---|---|
| `Send` API | Map-Reduce：一批独立任务并行扇出 | use-graph-api.mdx |
| `Command(goto=...)` | 一个返回值里同时改状态 + 定去向 | use-graph-api.mdx |
| `@entrypoint` / `@task` | 函数式 API：普通函数也能持久化/中断 | functional-api.mdx |

控制流是**纯 Python 机制**，对错与模型无关，所以用固定数据确定性复现；
生产里「产出主题」「写笑话」这类节点换成模型调用即可。

### 4.1 Demo 1：Send API —— Map-Reduce 并行扇出

第 1~3 节教的都是「一个节点跑完轮到下一个」的串行图。现实里常遇到
「一批互相独立的任务」——10 个文档各写摘要、5 个城市各查天气。
用 `Send` 是**运行时按数据量动态扇出**：有几条数据就起几个同名节点实例，并行跑，
结果靠 reducer 自动汇总。三个关键点（缺一个就跑不对）：

1. 汇总字段必须是 `Annotated[list, operator.add]` 这种**带 reducer** 的声明；
2. fan-out 函数返回 `[Send("节点名", 私有输入), ...]`，挂在条件边上；
3. worker 收到的 state 是 Send 第二参传进去的**私有输入**，不是完整 state。

In [ ]:
import operator
import time

from typing import Annotated, Literal

from typing_extensions import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.func import entrypoint, task
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send, interrupt


# ================================================================
# Demo 1：Send API —— Map-Reduce 与并行 fan-out
# ================================================================
class MapReduceState(TypedDict):
    """Map-Reduce 的公共状态。"""
    subjects: list[str]
    jokes: Annotated[list[str], operator.add]
    best: str


def generate_topics(state: MapReduceState) -> dict:
    """官方示例里这一步是「让模型生成主题」；本文件用固定清单保证可复现。"""
    print("[generate_topics] 产出 3 个主题（生产里这步是模型调用）")
    return {"subjects": ["狮子", "大象", "企鹅"]}


def fan_out_to_workers(state: MapReduceState) -> list[Send]:
    """条件边 = 分拣站：为 subjects 里每条数据各派一个 write_joke 实例。"""
    print(f"[fan_out] 把 {len(state['subjects'])} 个主题分发给 worker（并行执行）")
    return [Send("write_joke", {"subject": s}) for s in state["subjects"]]


def write_joke(state: dict) -> dict:
    """worker 节点：注意它收到的 state 只有 Send 传进来的 {"subject": ...}。"""
    subject = state["subject"]
    print(f"    [write_joke] 处理：{subject}")
    return {"jokes": [f"《{subject}》：冷笑话一则"]}


def pick_best(state: MapReduceState) -> dict:
    """reduce 之后的收口节点：这时才能看到全部 worker 的产出。"""
    print(f"[pick_best] 汇总到 {len(state['jokes'])} 条结果，挑第一条")
    return {"best": state["jokes"][0]}


map_reduce_graph = (
    StateGraph(MapReduceState)
    .add_node("generate_topics", generate_topics)
    .add_node("write_joke", write_joke)
    .add_node("pick_best", pick_best)
    .add_edge(START, "generate_topics")
    .add_conditional_edges("generate_topics", fan_out_to_workers, ["write_joke"])
    .add_edge("write_joke", "pick_best")
    .add_edge("pick_best", END)
    .compile()
)

In [ ]:
print("=" * 70)
print("Demo 1：Send API —— Map-Reduce 并行扇出")
print("=" * 70)
mr_result = map_reduce_graph.invoke({"subjects": [], "jokes": [], "best": ""})
print("\n最终状态：")
print("  subjects:", mr_result["subjects"])
for index, joke in enumerate(mr_result["jokes"], start=1):
    print(f"  jokes[{index}]:", joke)
print("  best    :", mr_result["best"])
print(
    "  ↑ 3 条结果**同时**在 jokes 里 —— 这就是 reducer(operator.add) 的作用；\n"
    "    如果把 MapReduceState 里的 jokes 改成裸 list（没有 Annotated 声明），\n"
    "    三个 worker 在**同一 super-step** 写同一字段会直接抛 InvalidUpdateError\n"
    "    （LastValue 通道报 At key 'jokes': Can receive only one value per step），图直接失败。\n"
    "    注意区分：跨 super-step 的串行写入才是「后写覆盖先写」。"
)

### 预期输出

```text
======================================================================
Demo 1：Send API —— Map-Reduce 并行扇出
======================================================================
[generate_topics] 产出 3 个主题（生产里这步是模型调用）
[fan_out] 把 3 个主题分发给 worker（并行执行）
    [write_joke] 处理：狮子
    [write_joke] 处理：大象
    [write_joke] 处理：企鹅
[pick_best] 汇总到 3 条结果，挑第一条

最终状态：
  subjects: ['狮子', '大象', '企鹅']
  jokes[1]: 《狮子》：冷笑话一则
  jokes[2]: 《大象》：冷笑话一则
  jokes[3]: 《企鹅》：冷笑话一则
  best    : 《狮子》：冷笑话一则
  ↑ 3 条结果**同时**在 jokes 里 —— 这就是 reducer(operator.add) 的作用；
    如果把 MapReduceState 里的 jokes 改成裸 list（没有 Annotated 声明），
    三个 worker 在**同一 super-step** 写同一字段会直接抛 InvalidUpdateError
    （LastValue 通道报 At key 'jokes': Can receive only one value per step），图直接失败。
    注意区分：跨 super-step 的串行写入才是「后写覆盖先写」。
```

### 4.2 Demo 2：Command(goto=...) —— 边改状态边决定去向

课案里的 `Command` 只用过 `Command(resume=...)`（中断恢复）。官方文档里 `Command`
还有 `update` / `goto` 两个字段：`Command(update={...}, goto="节点名")` 一个返回值里
**同时**改状态 + 定去向。什么时候用它替代条件边？——当「往哪走」和「状态怎么改」
本来就是同一件事的时候。

⚠️ 官方明确警告：`Command` 只**新增动态边**，不会取消静态边。如果 `decide` 同时
用 `add_edge` 连了别的节点，那么 goto 的目标和静态边的目标**都会被执行**。
所以下面的 `decide` 故意不声明任何静态出边。

In [ ]:
class RouteState(TypedDict):
    mode: str
    visited: Annotated[list[str], operator.add]


def decide(state: RouteState) -> Command[Literal["fast_path", "slow_path"]]:
    """返回类型注解里用 Literal 声明可能的去向（官方推荐写法，便于静态检查与画图）。"""
    if state["mode"] == "fast":
        return Command(update={"visited": ["decide → fast_path"]}, goto="fast_path")
    return Command(update={"visited": ["decide → slow_path"]}, goto="slow_path")


def fast_path(state: RouteState) -> dict:
    return {"visited": ["fast_path 执行完毕"]}


def slow_path(state: RouteState) -> dict:
    return {"visited": ["slow_path 执行完毕"]}


route_graph = (
    StateGraph(RouteState)
    .add_node("decide", decide)
    .add_node("fast_path", fast_path)
    .add_node("slow_path", slow_path)
    .add_edge(START, "decide")
    .add_edge("fast_path", END)
    .add_edge("slow_path", END)
    .compile()
)


class LoopState(TypedDict):
    n: int
    log: Annotated[list[str], operator.add]


def tick(state: LoopState) -> Command[Literal["tick", "__end__"]]:
    n = state.get("n", 0) + 1
    if n >= 3:
        return Command(update={"n": n, "log": [f"第 {n} 次：到点了，结束"]}, goto=END)
    return Command(update={"n": n, "log": [f"第 {n} 次：继续"]}, goto="tick")


loop_graph = (
    StateGraph(LoopState)
    .add_node("tick", tick)
    .add_edge(START, "tick")
    .compile()
)

In [ ]:
print("\n" + "=" * 70)
print("Demo 2：Command(goto=...) —— 边改状态边决定去向")
print("=" * 70)
for mode in ("fast", "slow"):
    result = route_graph.invoke({"mode": mode, "visited": []})
    print(f"  mode={mode!r} 最终状态: visited={result['visited']}")
print(
    "  ↑ decide 节点上**没有任何静态出边**，去向全靠 Command(goto=...) 动态决定；\n"
    "    对比课案的条件边写法：那需要单独再写一个路由函数，Command 把它和改状态合并了。"
)

loop_result = loop_graph.invoke({"n": 0, "log": []})
print(f"\n  循环演示（Command 自跳，n 到 3 停）：n={loop_result['n']} log={loop_result['log']}")
print("  ↑ 图上没有 tick→tick 的循环边，循环是 Command(goto='tick') 自己跳出来的")

### 预期输出

```text

======================================================================
Demo 2：Command(goto=...) —— 边改状态边决定去向
======================================================================
  mode='fast' 最终状态: visited=['decide → fast_path', 'fast_path 执行完毕']
  mode='slow' 最终状态: visited=['decide → slow_path', 'slow_path 执行完毕']
  ↑ decide 节点上**没有任何静态出边**，去向全靠 Command(goto=...) 动态决定；
    对比课案的条件边写法：那需要单独再写一个路由函数，Command 把它和改状态合并了。

  循环演示（Command 自跳，n 到 3 停）：n=3 log=['第 1 次：继续', '第 2 次：继续', '第 3 次：到点了，结束']
  ↑ 图上没有 tick→tick 的循环边，循环是 Command(goto='tick') 自己跳出来的
```

### 4.3 Demo 3：函数式 API —— @entrypoint / @task + 中断 + 重放不重算

官方有**两种**建模方式，课案 100% 只教了图 API（StateGraph）：

- **图 API** —— 显式画节点和边，状态机思路，适合复杂分支/多人协作/可视化调试；
- **函数式 API** —— 普通 Python 函数 + 两个装饰器，适合计算型流程与快速原型。

两者共用同一套运行时（checkpointer / interrupt / 流式），可以混用。两个装饰器：

- `@task` = 可持久化的函数：返回值写进 checkpoint，**恢复时不重算**；
- `@entrypoint` = 可持久化的流程入口：支持 checkpointer、interrupt、stream。

本 Demo 把「函数式 API + 人工中断 + 重放不重算」串成一条：第一次 invoke 跑到
`interrupt()` 暂停 → resume 继续 → 前面那个昂贵的 `@task` **不会**再执行第二次。

In [ ]:
task_calls = {"n": 0}


@task
def expensive_double(x: int) -> int:
    """模拟昂贵计算：调用次数会被打印出来，用来证明「恢复时有没有重算」。"""
    task_calls["n"] += 1
    print(f"    [task expensive_double] 第 {task_calls['n']} 次真正执行（模拟耗时计算）")
    time.sleep(0.3)
    return x * 2


@entrypoint(checkpointer=MemorySaver())
def review_flow(inp: dict) -> dict:
    """函数式 API 的入口：普通函数写法，靠装饰器获得持久化与中断能力。"""
    data = expensive_double(inp["x"]).result()
    decision = interrupt({"question": "翻倍完成，放行吗？", "data": data})
    return {"data": data, "decision": decision}

In [ ]:
config = {"configurable": {"thread_id": "func-demo-1"}}

print("\n" + "=" * 70)
print("Demo 3：函数式 API —— @entrypoint / @task + 中断 + 重放不重算")
print("=" * 70)
print("\n--- 第一次 invoke：跑到 interrupt() 就暂停 ---")
first = review_flow.invoke({"x": 21}, config)
print("  返回类型:", type(first).__name__)
print("  返回内容:", first)
print("  ↑ 图停在这里，还没返回最终结果；interrupt() 收到的值存在 __interrupt__ 里")

### 预期输出

```text

======================================================================
Demo 3：函数式 API —— @entrypoint / @task + 中断 + 重放不重算
======================================================================

--- 第一次 invoke：跑到 interrupt() 就暂停 ---
    [task expensive_double] 第 1 次真正执行（模拟耗时计算）
  返回类型: dict
  返回内容: {'__interrupt__': [Interrupt(value={'question': '翻倍完成，放行吗？', 'data': 42}, id='4ce2ff41211200463d698fbb59c0ad47')]}
  ↑ 图停在这里，还没返回最终结果；interrupt() 收到的值存在 __interrupt__ 里
```

> ⚠️ 这一格 `返回内容` 里的 `__interrupt__` 带一个随机 `id`，每次运行都不同，别逐字比对。

In [ ]:
print("\n--- resume：用 Command(resume=...) 带着决定恢复 ---")
second = review_flow.invoke(Command(resume="放行"), config)
print("  返回内容:", second)
print(
    f"  expensive_double 累计真正执行 {task_calls['n']} 次"
    " ← 恢复时**没有重算**（结果直接从 checkpoint 复用，这就是 @task 的价值）"
)

### 预期输出

```text

--- resume：用 Command(resume=...) 带着决定恢复 ---
  返回内容: {'data': 42, 'decision': '放行'}
  expensive_double 累计真正执行 1 次 ← 恢复时**没有重算**（结果直接从 checkpoint 复用，这就是 @task 的价值）
```

In [ ]:
print("\n--- 换个 thread_id：一切重新开始 ---")
other = review_flow.invoke({"x": 5}, {"configurable": {"thread_id": "func-demo-2"}})
print("  返回内容:", other)
print(
    f"  expensive_double 累计真正执行 {task_calls['n']} 次"
    " ← 新线程会重算（@task 的复用是**按 thread 隔离**的）"
)
print("\n全部 Demo 执行完毕（0 次模型调用，离线可复现）。")

### 预期输出

```text

--- 换个 thread_id：一切重新开始 ---
    [task expensive_double] 第 2 次真正执行（模拟耗时计算）
  返回内容: {'__interrupt__': [Interrupt(value={'question': '翻倍完成，放行吗？', 'data': 10}, id='92ce76f6b5df75d963ecd085fc38a685')]}
  expensive_double 累计真正执行 2 次 ← 新线程会重算（@task 的复用是**按 thread 隔离**的）

全部 Demo 执行完毕（0 次模型调用，离线可复现）。
```

> ⚠️ 这一格 `返回内容` 里的 `__interrupt__` 带一个随机 `id`，每次运行都不同，别逐字比对。

## 小结

- **三层框架**：`DeepAgents ⊃ create_agent ⊃ 内部拼出的 StateGraph`；越往上越省事、越往下越自由；
- **状态**是 `TypedDict`，字段声明同时决定了「有没有」和「怎么合并」；
- **节点**是普通函数，**返回增量而不是全量**；
- **普通边**写死跳转，**条件边**由路由函数的返回值决定（`Send` 是条件边的动态并行版）；
- **`Command(goto)`** 把「改状态 + 定去向」合成一次返回；**`Command(resume)`** 恢复中断；
- **reducer**（`Annotated[..., operator.add]`）让「追加」变成声明式，也是 `Send` 并行汇总的根基；
- **函数式 API**（`@entrypoint` / `@task`）是图 API 之外的另一种建模方式，`@task` 恢复时不重算。

下一课 `02_记忆_短期与长期.ipynb` 会看到：把 `operator.add` 换成 LangGraph 内置的
`add_messages`，再加一个 checkpointer，图就**记得住上一轮对话**了。

## 常见坑

1. **节点返回的必须是 dict（增量）**，不是完整 state。返回整个 state 不会报错，
   但 reducer 的语义会变得难以预料。
2. **`compile()` 之后图结构就冻结了**：再 `add_node` / `add_edge` 不会生效、
   **也不会报错** —— 要改结构必须改 builder 后重新 `compile()`。
3. **条件边的路由函数不执行跳转**，它只返回名字；跳转靠映射表。
4. **状态字段没声明却返回同名键，会被静默丢弃**（TypedDict 不做运行时校验）。
5. **`Send` 的 worker 收到的是私有输入**（Send 第二参），不是完整 state。
6. **并行写同一字段必须带 reducer**：同一 super-step 内写多次会抛 `InvalidUpdateError`。
7. **`Command` 只加动态边**：同时存在静态出边时两条路都会跑。
8. **`interrupt()` 不能被 try/except 包住**，否则框架的暂停信号会被吞掉。
9. **`@task` 只能在 `@entrypoint` 内调用**；`@task` 的复用按 `thread_id` 隔离。

## 官方链接

- 图 API（StateGraph / Node / Edge / Command）：<https://docs.langchain.com/oss/python/langgraph/graph-api>
- 快速上手：<https://docs.langchain.com/oss/python/langgraph/quickstart>
- 用图 API 构建工作流：<https://docs.langchain.com/oss/python/langgraph/use-graph-api>
- 函数式 API：<https://docs.langchain.com/oss/python/langgraph/functional-api>
- 两种 API 选型：<https://docs.langchain.com/oss/python/langgraph/choosing-apis>
- 六种经典工作流模式：<https://docs.langchain.com/oss/python/langgraph/workflows-agents>